<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/02_build_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Ensure Google Drive is mounted
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Re-clone repository if missing
%cd /content
if not os.path.exists("/content/DP-MMFL"):
    try:
        token = userdata.get("GITHUB_TOKEN")
        !git clone https://{token}@github.com/PreethamHD/DP-MMFL.git
    except Exception:
        !git clone https://github.com/PreethamHD/DP-MMFL.git

# 3. Verify labels.py exists on disk
labels_file = Path("/content/DP-MMFL/src/dp_mmfl/data/labels.py")
print("labels.py exists:", labels_file.exists())

# 4. Insert src at the head of sys.path
REPO_SRC = "/content/DP-MMFL/src"
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)

# 5. Test import
from dp_mmfl.data.labels import TARGET_COLUMNS, ALL_LABEL_COLUMNS
print("Import successful! Target count:", len(TARGET_COLUMNS))

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 66 (delta 18), reused 50 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 34.94 KiB | 11.65 MiB/s, done.
Resolving deltas: 100% (18/18), done.
labels.py exists: True
Import successful! Target count: 13


In [2]:
from pathlib import Path
import sys
import os
import pandas as pd
import numpy as np

# Ensure src is discoverable
REPO_SRC = "/content/DP-MMFL/src"
if REPO_SRC not in sys.path:
    sys.path.append(REPO_SRC)

from dp_mmfl.data.labels import (
    TARGET_COLUMNS,
    ALL_LABEL_COLUMNS,
    load_label_json,
    apply_label_policy,
)

# Persistent Drive storage locations
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

# Support both case-sensitive naming conventions
raw_candidates = [
    DRIVE_ROOT / "data" / "raw" / "CheXpertPlus",
    DRIVE_ROOT / "data" / "raw" / "chexpert_plus"
]
DATASET_ROOT = next((p for p in raw_candidates if p.exists()), raw_candidates[0])

CSV_PATH = DATASET_ROOT / "df_chexpert_plus_240401.csv"
REPORT_LABELS_PATH = DATASET_ROOT / "report_fixed.json"

PROCESSED_ROOT = DRIVE_ROOT / "data" / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = PROCESSED_ROOT / "chexpert_plus_manifest.parquet"

print(f"Dataset root:   {DATASET_ROOT}")
print(f"CSV path:       {CSV_PATH} (Exists: {CSV_PATH.exists()})")
print(f"Report labels:  {REPORT_LABELS_PATH} (Exists: {REPORT_LABELS_PATH.exists()})")
print(f"Target parquet: {MANIFEST_PATH}")

Dataset root:   /content/drive/MyDrive/DP-MMFL/data/raw/chexpert_plus
CSV path:       /content/drive/MyDrive/DP-MMFL/data/raw/chexpert_plus/df_chexpert_plus_240401.csv (Exists: True)
Report labels:  /content/drive/MyDrive/DP-MMFL/data/raw/chexpert_plus/report_fixed.json (Exists: True)
Target parquet: /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet


In [3]:
# Load core metadata table
df = pd.read_csv(CSV_PATH)
print("CSV shape:", df.shape)

# Load CheXbert report labels (line-delimited JSON)
report_labels_df = load_label_json(REPORT_LABELS_PATH)
print("Report labels shape:", report_labels_df.shape)

CSV shape: (223462, 27)
Report labels shape: (223462, 15)


In [4]:
csv_paths = set(df["path_to_image"])
label_paths = set(report_labels_df["path_to_image"])

missing_from_labels = len(csv_paths - label_paths)
missing_from_csv = len(label_paths - csv_paths)
dup_csv_paths = df["path_to_image"].duplicated().sum()
dup_label_paths = report_labels_df["path_to_image"].duplicated().sum()

print("Verification Checks:")
print(f"  CSV total paths:           {len(csv_paths)}")
print(f"  Label total paths:         {len(label_paths)}")
print(f"  Missing from labels:       {missing_from_labels}")
print(f"  Missing from CSV:          {missing_from_csv}")
print(f"  Duplicate paths in CSV:    {dup_csv_paths}")
print(f"  Duplicate paths in labels: {dup_label_paths}")

assert missing_from_labels == 0, f"Error: {missing_from_labels} CSV paths missing in labels."
assert missing_from_csv == 0, f"Error: {missing_from_csv} label paths missing in CSV."
assert dup_csv_paths == 0, "Error: Duplicate paths found in master CSV."
assert dup_label_paths == 0, "Error: Duplicate paths found in report labels."
print("\nAll key integrity checks passed.")

Verification Checks:
  CSV total paths:           223462
  Label total paths:         223462
  Missing from labels:       0
  Missing from CSV:          0
  Duplicate paths in CSV:    0
  Duplicate paths in labels: 0

All key integrity checks passed.


In [5]:
# 1. Merge report labels onto main manifest
label_subset = report_labels_df[["path_to_image"] + ALL_LABEL_COLUMNS].copy()
manifest = df.merge(
    label_subset,
    on="path_to_image",
    how="left",
    validate="one_to_one",
)
print("Merged manifest shape:", manifest.shape)

# 2. Compute targets and binary supervision masks
targets, masks = apply_label_policy(manifest, TARGET_COLUMNS)

targets = targets.rename(columns={label: f"target_{label}" for label in TARGET_COLUMNS})
masks = masks.rename(columns={label: f"mask_{label}" for label in TARGET_COLUMNS})

# 3. Concatenate target and mask frames
manifest = pd.concat([manifest, targets, masks], axis=1)

Merged manifest shape: (223462, 41)


In [6]:
MANIFEST_COLUMNS = [
    # Identifiers
    "path_to_image",
    "path_to_dcm",
    "deid_patient_id",

    # Text Modality
    "report",

    # Demographics & Split
    "age",
    "sex",
    "race",
    "ethnicity",
    "split",
]

# Raw labels, binary targets, and supervision masks
MANIFEST_COLUMNS += ALL_LABEL_COLUMNS
MANIFEST_COLUMNS += [f"target_{label}" for label in TARGET_COLUMNS]
MANIFEST_COLUMNS += [f"mask_{label}" for label in TARGET_COLUMNS]

manifest = manifest[MANIFEST_COLUMNS].copy()

# Add persistent unique index
manifest.insert(0, "sample_id", range(len(manifest)))
print("Cleaned manifest shape:", manifest.shape)

Cleaned manifest shape: (223462, 50)


In [7]:
# Verify mask/target assignments against raw values
for label in TARGET_COLUMNS:
    raw = manifest[label]
    target = manifest[f"target_{label}"]
    mask = manifest[f"mask_{label}"]

    valid_pos = raw == 1.0
    valid_neg = raw == 0.0
    ignored = raw.isna() | (raw == -1.0)

    assert (mask[valid_pos] == 1).all()
    assert (target[valid_pos] == 1.0).all()

    assert (mask[valid_neg] == 1).all()
    assert (target[valid_neg] == 0.0).all()

    assert (mask[ignored] == 0).all()
    assert target[ignored].isna().all()

print("Strict label policy verification passed.")

print("\n--- Integrity Summary ---")
print(f"Total Rows:           {len(manifest)}")
print(f"Unique Patients:      {manifest['deid_patient_id'].nunique()}")
print(f"Missing Patient IDs:  {manifest['deid_patient_id'].isna().sum()}")
print(f"Duplicate Sample IDs: {manifest['sample_id'].duplicated().sum()}")
print(f"Duplicate Paths:      {manifest['path_to_image'].duplicated().sum()}")

print("\n--- Split Breakdown ---")
print(manifest["split"].value_counts(dropna=False))

Strict label policy verification passed.

--- Integrity Summary ---
Total Rows:           223462
Unique Patients:      64725
Missing Patient IDs:  0
Duplicate Sample IDs: 0
Duplicate Paths:      0

--- Split Breakdown ---
split
train    223228
valid       234
Name: count, dtype: int64


In [8]:
supervision_summary = []
for label in TARGET_COLUMNS:
    mask = manifest[f"mask_{label}"]
    supervision_summary.append({
        "Condition": label,
        "Supervised Samples": int(mask.sum()),
        "Ignored Samples": int((mask == 0).sum()),
        "Supervision Rate (%)": round(float(mask.mean()) * 100, 2),
    })

supervision_df = pd.DataFrame(supervision_summary).sort_values("Supervision Rate (%)")
print(supervision_df.to_string(index=False))

                 Condition  Supervised Samples  Ignored Samples  Supervision Rate (%)
             Pleural Other                5810           217652                  2.60
                 Pneumonia               10683           212779                  4.78
               Lung Lesion               14622           208840                  6.54
                  Fracture               15737           207725                  7.04
               Atelectasis               39028           184434                 17.47
Enlarged Cardiomediastinum               41862           181600                 18.73
             Consolidation               47971           175491                 21.47
              Cardiomegaly               62168           161294                 27.82
                     Edema               77326           146136                 34.60
              Pneumothorax               86354           137108                 38.64
              Lung Opacity              117065        

In [9]:
# Install pyarrow if not already installed
!pip install -q pyarrow

# Write to Drive
manifest.to_parquet(MANIFEST_PATH, index=False)
print(f"Parquet manifest successfully written to: {MANIFEST_PATH}")

# Reload and check round-trip integrity
manifest_check = pd.read_parquet(MANIFEST_PATH)
assert manifest_check.shape == manifest.shape, "Shape mismatch after reloading!"
assert list(manifest_check.columns) == list(manifest.columns), "Column schema mismatch!"

print("Round-trip validation passed.")
print("Reloaded shape:", manifest_check.shape)

Parquet manifest successfully written to: /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Round-trip validation passed.
Reloaded shape: (223462, 50)


In [10]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"

manifest = pd.read_parquet(MANIFEST_PATH)
print("Shape:", manifest.shape)
print(manifest.head())

Shape: (223462, 50)
   sample_id                                path_to_image  \
0          0  train/patient00003/study1/view1_frontal.jpg   
1          1  train/patient00007/study2/view1_frontal.jpg   
2          2  train/patient00007/study1/view1_frontal.jpg   
3          3  train/patient00009/study1/view2_lateral.jpg   
4          4  train/patient00009/study1/view1_frontal.jpg   

                                   path_to_dcm deid_patient_id  \
0  train/patient00003/study1/view1_frontal.dcm    patient00003   
1  train/patient00007/study2/view1_frontal.dcm    patient00007   
2  train/patient00007/study1/view1_frontal.dcm    patient00007   
3  train/patient00009/study1/view2_lateral.dcm    patient00009   
4  train/patient00009/study1/view1_frontal.dcm    patient00009   

                                              report   age   sex   race  \
0  NARRATIVE:\nCHEST, ONE VIEW: 2-10-2001\nFINDIN...  41.0  Male  White   
1  NARRATIVE:\nChest 1 View: July 20\n \nHISTORY:...  69.0  Male  

In [11]:
images_per_patient = manifest.groupby("deid_patient_id").size()

print("--- Images Per Patient Summary ---")
print(images_per_patient.describe())

print("\nPatients with exactly 1 image:", (images_per_patient == 1).sum())
print("Patients with >1 image:       ", (images_per_patient > 1).sum())
print("Maximum images for one patient:", images_per_patient.max())

print("\n--- Image Count Distribution (First 20 Frequencies) ---")
print(images_per_patient.value_counts().sort_index().head(20))

--- Images Per Patient Summary ---
count    64725.000000
mean         3.452484
std          4.651210
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         92.000000
dtype: float64

Patients with exactly 1 image: 22766
Patients with >1 image:        41959
Maximum images for one patient: 92

--- Image Count Distribution (First 20 Frequencies) ---
1     22766
2     17563
3      6960
4      4688
5      2913
6      2144
7      1417
8      1163
9       908
10      688
11      554
12      429
13      353
14      300
15      242
16      197
17      154
18      175
19      129
20      127
Name: count, dtype: int64


In [12]:
train_patients = set(manifest.loc[manifest["split"] == "train", "deid_patient_id"])
valid_patients = set(manifest.loc[manifest["split"] == "valid", "deid_patient_id"])
overlap = train_patients & valid_patients

print(f"Train patients:  {len(train_patients):,}")
print(f"Valid patients:  {len(valid_patients):,}")
print(f"Patient overlap: {len(overlap)}")

assert len(overlap) == 0, f"DATA LEAKAGE DETECTED: {len(overlap)} patients overlap between splits!"
print("Official splits are strictly patient-disjoint.")

Train patients:  64,525
Valid patients:  200
Patient overlap: 0
Official splits are strictly patient-disjoint.


In [13]:
patient_split_counts = manifest.groupby("deid_patient_id")["split"].nunique()
print("Split assignment multiplicity counts (expecting solely 1):")
print(patient_split_counts.value_counts())

mixed_split_patients = patient_split_counts[patient_split_counts > 1]
print(f"Patients appearing across multiple splits: {len(mixed_split_patients)}")

Split assignment multiplicity counts (expecting solely 1):
split
1    64725
Name: count, dtype: int64
Patients appearing across multiple splits: 0


In [14]:
split_summary = manifest.groupby("split").agg(
    images=("sample_id", "count"),
    patients=("deid_patient_id", "nunique"),
)
print("--- Official Split Summary ---")
print(split_summary)

--- Official Split Summary ---
       images  patients
split                  
train  223228     64525
valid     234       200


In [15]:
split_patient_images = (
    manifest.groupby(["split", "deid_patient_id"])
    .size()
    .reset_index(name="image_count")
)

print("--- Image Count Summary per Patient by Split ---")
print(split_patient_images.groupby("split")["image_count"].describe())

--- Image Count Summary per Patient by Split ---
         count      mean       std  min  25%  50%  75%   max
split                                                       
train  64525.0  3.459558  4.656617  1.0  1.0  2.0  4.0  92.0
valid    200.0  1.170000  0.414680  1.0  1.0  1.0  1.0   3.0


In [16]:
print("--- Sample Image Relative Paths ---")
print(manifest["path_to_image"].head(20).to_string(index=False))

print("\n--- View Column Check ---")
print(
    manifest["frontal_lateral"]
    if "frontal_lateral" in manifest.columns
    else "frontal_lateral was not retained in canonical manifest"
)

--- Sample Image Relative Paths ---
train/patient00003/study1/view1_frontal.jpg
train/patient00007/study2/view1_frontal.jpg
train/patient00007/study1/view1_frontal.jpg
train/patient00009/study1/view2_lateral.jpg
train/patient00009/study1/view1_frontal.jpg
train/patient00016/study1/view2_lateral.jpg
train/patient00016/study1/view1_frontal.jpg
train/patient00021/study1/view1_frontal.jpg
train/patient00026/study1/view1_frontal.jpg
train/patient00026/study1/view2_lateral.jpg
train/patient00027/study1/view2_lateral.jpg
train/patient00027/study1/view1_frontal.jpg
train/patient00031/study1/view1_frontal.jpg
train/patient00031/study3/view1_frontal.jpg
train/patient00031/study2/view1_frontal.jpg
train/patient00034/study1/view1_frontal.jpg
train/patient00038/study1/view1_frontal.jpg
train/patient00038/study3/view1_frontal.jpg
train/patient00038/study2/view1_frontal.jpg
train/patient00044/study6/view2_lateral.jpg

--- View Column Check ---
frontal_lateral was not retained in canonical manifest


In [17]:
TARGET_COLUMNS = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

patient_label_counts = pd.DataFrame({
    label: manifest.groupby("deid_patient_id")[f"mask_{label}"].sum()
    for label in TARGET_COLUMNS
})

print("--- Supervised Mask Count per Patient ---")
print(patient_label_counts.describe().T[["mean", "std", "min", "50%", "max"]])

--- Supervised Mask Count per Patient ---
                                mean       std  min  50%   max
Enlarged Cardiomediastinum  0.646767  1.113367  0.0  0.0  22.0
Cardiomegaly                0.960494  1.733094  0.0  0.0  47.0
Lung Opacity                1.808652  2.911274  0.0  1.0  63.0
Lung Lesion                 0.225910  0.704883  0.0  0.0  26.0
Edema                       1.194685  2.184458  0.0  1.0  60.0
Consolidation               0.741151  1.286738  0.0  0.0  26.0
Pneumonia                   0.165052  0.538743  0.0  0.0  15.0
Atelectasis                 0.602982  1.186601  0.0  0.0  25.0
Pneumothorax                1.334168  2.115929  0.0  1.0  55.0
Pleural Effusion            2.132592  3.254320  0.0  1.0  69.0
Pleural Other               0.089764  0.435019  0.0  0.0  15.0
Fracture                    0.243136  0.709515  0.0  0.0  21.0
Support Devices             2.094415  3.506358  0.0  1.0  80.0


In [18]:
print("--- Top 20 Most Prolific Patients ---")
print(images_per_patient.sort_values(ascending=False).head(20))

print("\nMean images/patient:  ", images_per_patient.mean())
print("Median images/patient:", images_per_patient.median())

--- Top 20 Most Prolific Patients ---
deid_patient_id
patient33155    92
patient28746    92
patient04462    89
patient24163    86
patient34615    85
patient19317    80
patient13011    79
patient05702    79
patient14282    77
patient10970    76
patient03122    76
patient26381    76
patient06028    76
patient20479    75
patient30627    74
patient09793    73
patient13162    72
patient19689    72
patient31471    72
patient24428    71
dtype: int64

Mean images/patient:   3.4524835843955195
Median images/patient: 2.0


In [19]:
from google.colab import userdata

%cd /content/DP-MMFL

!git config --global user.name "PreethamHD"
!git config --global user.email "preethamgowda837@gmail.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    !git remote set-url origin https://{token}@github.com/PreethamHD/DP-MMFL.git
except Exception:
    pass

!git add src/ configs/ docs/ .gitignore
!git commit -m "build manifest"
!git push origin main
!git status

/content/DP-MMFL
[main fa9eb47] build manifest
 1 file changed, 0 insertions(+), 0 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 473 bytes | 473.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   78d022b..fa9eb47  main -> main
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
